# 🧪 Experiment EIC_083 — Feature-Adapter Hybrid PPO vs Baseline


## 📝 Introduction

In this experiment, we investigate how the same `HybridPpoPolicy` backbone behaves when we make its input information **explicit** through feature adapters. In particular, we compare a raw-state PPO, a raw-state-plus-last-demand PPO, and a raw-state-plus-forecast-path PPO against the baseline heuristic `OrderUpToPolicy`.

This notebook is the first constant-case showcase for the new feature-adapter refactor. The key teaching question is not only whether PPO can beat the baseline, but also how the chosen **feature set** changes policy quality.


## 📦 Imports

Here we define all required imports to conduct our experiment.

In [ ]:
# Imports
import numpy as np
from pprint import pprint

# Optional: in-notebook sanity check (no-op when installed)
from inventory.evaluation.notebooks.bootstrap import ensure_inventory_imports
ensure_inventory_imports(verbose=False)

# Evaluation
from inventory.evaluation import make_eval_info
from inventory.evaluation.reporting import summarize_totals

# Dynamic system framework
from inventory.core.dynamics import DynamicSystem

# Problem definitions
from inventory.problems.inventory_mvp import inventory_cost, inventory_cost_extended, inventory_transition, inventory_transition_capped_1d, inventory_cost_extended_capped

# Demand model (exogenous)
from inventory.problems.demand_models import PoissonConstantDemand

# Forecaster
from inventory.forecasters.ml import MlDemandForecaster, ConstantFeatureAdapter
from inventory.forecasters.naive import ConstantMeanForecaster,NaiveForecaster

# Policies and feature adapters
from inventory.policies.baselines import OrderUpToPolicy
from inventory.policies.ppo import HybridPpoPolicy, PPOHyperParams, train_ppo_with_eval_gate
from inventory.policies.policy_features import (
    ForecastPathFeatureAdapter,
    LastDemandFeatureAdapter,
    RawStateFeatureAdapter,
)


---

## 🚧 Problem definition

In this experiment we use our benchmark inventory problem with an constant Poisson demand and our standard set of cost parameters. Keeping this problem definition constant across experiments enables us to compare the effects of different policies.

In [ ]:
# === Defining the dynamic system ===

# Simulation horizon
T = 30

# Initial Inventory
S0 = np.array([150.0], dtype=float)

# Constraints
x_max = 480
dx = 10
S_max = 480

# Exogenous demand
exo = PoissonConstantDemand(lam=300)

# Cost parameters 
p, c, h, b, K = 2.0, 0.5, 0.03, 1.0, 10.0

# Transition function
#transition_func = inventory_transition
transition_func = lambda s, a, w, t: inventory_transition_capped_1d(s, a, w, t, s_max=S_max)

# Cost functions
#cost_func = lambda s, a, w, t: inventory_cost(s, a, w, t, p=p, c=c, h=h, b=b)
#cost_func = lambda s, a, w, t: inventory_cost_extended(s, a, w, t, p=p, c=c, h=h, b=b, K=K)
cost_func = lambda s, a, w, t: inventory_cost_extended_capped(s, a, w, t, p=p, c=c, h=h, b=b, K=K, s_max=S_max)

# Dynamic System
system = DynamicSystem(
    transition_func=transition_func,
    cost_func=cost_func,
    exogenous_model=exo,
    sim_seed=42,
    d_s=1,
    d_x=1,
    d_w=1,
 )

# Strict-CRN Monte Carlo evaluation settings
seed0 = 1234
n_episodes = 200
eval_info = make_eval_info(T=T, det_mode='argmax')
eval_info

To get an idea about the kind of demand pattern we are dealing with in this problem setting, let us have a lookt at one expamplary demand that matches the CRN “reference episode” seed stream.

In [ ]:
from inventory.evaluation.plotting import plot_seasonal_reference_episode_sample_path

_ = plot_seasonal_reference_episode_sample_path(
    exo,T,seed0=seed0,n_episodes=n_episodes,figsize=(14, 7),
    )

---

## 🔮 Forecaster defintion

In this section, we specify the information model used by the **forecast-path feature adapter**. The raw-state and last-demand PPO variants do not consume this forecaster.


In [ ]:
# === Naive Forecaster ===
# Naive forecaster: repeat the last observed demand for the next H steps
forecaster_naive = NaiveForecaster(default_value=float(exo.lam))

# Print example prediction: Planned info payload passed into the CFA policy / forecaster call
naive_info_demo = {"last_demand": [295.0],}
H_demo = 5
mu_demo_naive = forecaster_naive.forecast_mean_path(S0, t=0, H=H_demo, info=naive_info_demo)
print("naive_info_demo:", naive_info_demo)
print(f"mu_demo_naive (H={H_demo}):", np.round(mu_demo_naive, 2))

In [ ]:
# === ML Forecaster ===
# Here we use the `MlDemandForecaster` using `trees` with a `ConstantFeatureAdapter` to learn a simple constant demand model.
adapter = ConstantFeatureAdapter(exo)
forecaster_ml = MlDemandForecaster(
    adapter, 
    model_type="tree", 
    tree_max_depth=6,
    min_samples_leaf=20,
    random_state=0)

# Train on synthetic data generated from the constant-demand exogenous model
fit_seed = 7
val_seed = 99

forecaster_ml.fit_from_exogenous(
    n_samples=60000,
    seed=fit_seed,
    t_start=0,
    val_samples=1000,
    val_seed=val_seed,
    val_t_start=8000,
 )

# Print ML forecaster fit report summary
print("ML forecaster fit_report (summary):")
print("  model_type:", forecaster_ml.fit_report.get("model_type"))
print("  feature_dim:", forecaster_ml.fit_report.get("feature_dim"))
for split in ("train", "val"):
    rep = forecaster_ml.fit_report.get(split, {})
    mae = rep.get("MAE")
    rmse = rep.get("RMSE")
    bias = rep.get("Bias")
    n = rep.get("n_samples")
    print(f"  {split}: n={n} | MAE={mae:.3f} | RMSE={rmse:.3f} | Bias={bias:.3f}")

# Print example prediction: forecast mean demand path
H_demo = 5
mu_demo = forecaster_ml.forecast_mean_path(S0, t=0, H=H_demo)
print(f"\nmu_demo (H={H_demo}):", np.round(mu_demo, 2))

---

## 🤖 Policy definition

In this section, we define the policies under investigation. The baseline remains unchanged, while the PPO comparison now holds the PPO backbone fixed and varies only the explicit observation adapter.


In [ ]:
# === Baseline Policy ===
# Baseline: a fixed order-up-to target
baseline = OrderUpToPolicy(target_level=350.0, x_max=x_max, dx=dx)


In [ ]:
# === PPO feature-adapter setup ===
# We keep the PPO backbone fixed and vary only the explicit observation adapter.

profile = "fast"  # "fast" | "quality"

if profile == "fast":
    ppo_hparams = PPOHyperParams(lr=3e-4, n_epochs=3, minibatch_size=256)
    total_train_episodes = 600
    chunk_episodes = 100
    gate_eval_episodes = min(100, int(n_episodes))
elif profile == "quality":
    ppo_hparams = PPOHyperParams(lr=3e-4, n_epochs=10, minibatch_size=256)
    total_train_episodes = 1200
    chunk_episodes = 200
    gate_eval_episodes = int(n_episodes)
else:
    raise ValueError("Unknown profile")

# Device selection: prefer MPS on macOS if available
device = "cpu"
try:
    import torch
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
except Exception:
    torch = None  # torch might not be installed; HybridPpoPolicy will error on init

state_feature_names = ["inventory"]
do_train = True


def build_ppo(feature_adapter, *, hidden=128, seed=0):
    return HybridPpoPolicy(
        d_s=int(S0.shape[0]),
        s_max=int(S_max),
        x_max=int(x_max),
        dx=int(dx),
        hidden=int(hidden),
        device=device,
        hparams=ppo_hparams,
        seed=int(seed),
        deterministic_eval=False,
        feature_adapter=feature_adapter,
    )


def maybe_train(label, ppo, *, train_seed_offset=0):
    if not do_train:
        return []
    hist = train_ppo_with_eval_gate(
        system=system,
        ppo=ppo,
        baseline_policy=baseline,
        S0=S0,
        T=int(T),
        total_train_episodes=int(total_train_episodes),
        chunk_episodes=int(chunk_episodes),
        eval_episodes=int(gate_eval_episodes),
        eval_seed0=int(seed0) + 999,
        train_seed0=int(seed0) + int(train_seed_offset),
        eval_info=eval_info,
        verbose=True,
    )
    print(f"Last gate record ({label}):")
    print(hist[-1] if hist else None)
    return hist


In [ ]:
# === Feature-adapter PPO variants ===
forecast_horizon = 3
forecast_demand_scale = 100.0
last_demand_scale = 100.0

try:
    ppo_raw_state = build_ppo(
        RawStateFeatureAdapter(
            raw_state_dim=int(S0.shape[0]),
            state_feature_names=state_feature_names,
        ),
        hidden=128,
        seed=0,
    )
    ppo_last_demand = build_ppo(
        LastDemandFeatureAdapter(
            raw_state_dim=int(S0.shape[0]),
            demand_scale=float(last_demand_scale),
            state_feature_names=state_feature_names,
        ),
        hidden=128,
        seed=1,
    )
    ppo_forecast = build_ppo(
        ForecastPathFeatureAdapter(
            forecaster=forecaster_ml,
            raw_state_dim=int(S0.shape[0]),
            horizon=int(forecast_horizon),
            demand_scale=float(forecast_demand_scale),
            state_feature_names=state_feature_names,
        ),
        hidden=128,
        seed=2,
    )
except ImportError as e:
    raise RuntimeError(
        "HybridPpoPolicy needs PyTorch. Install it into your environment (e.g. `pip install torch`)."
    ) from e

history_raw_state = maybe_train("raw-state PPO", ppo_raw_state, train_seed_offset=0)
history_last_demand = maybe_train("last-demand PPO", ppo_last_demand, train_seed_offset=50_000)
history_forecast = maybe_train("forecast-path PPO", ppo_forecast, train_seed_offset=100_000)

feature_probe_info = {"last_demand": 310.0}
feature_summaries = {
    "ppo_raw_state": {
        "describe": ppo_raw_state.describe_features(),
        "feature_names": ppo_raw_state.feature_names(),
        "sample_obs": ppo_raw_state._observation_vector(S0, t=0, info=None).round(3).tolist(),
    },
    "ppo_last_demand": {
        "describe": ppo_last_demand.describe_features(),
        "feature_names": ppo_last_demand.feature_names(),
        "sample_obs": ppo_last_demand._observation_vector(S0, t=1, info=feature_probe_info).round(3).tolist(),
    },
    "ppo_forecast": {
        "describe": ppo_forecast.describe_features(),
        "feature_names": ppo_forecast.feature_names(),
        "sample_obs": ppo_forecast._observation_vector(S0, t=0, info=None).round(3).tolist(),
    },
}

pprint(feature_summaries)


In [ ]:
# === Standalone breakdown for ppo_raw_state ===
# This cell repeats only the raw-state PPO setup so you can inspect that variant
# on its own. If you already ran the combined feature-adapter cell above, rerunning
# this cell will simply rebuild and retrain `ppo_raw_state`.


# === Feature-adapter PPO variants ===
forecast_horizon = 3
forecast_demand_scale = 100.0
last_demand_scale = 100.0

raw_state_adapter = RawStateFeatureAdapter(
    raw_state_dim=int(S0.shape[0]),
    state_feature_names=state_feature_names,
)

# Train the newly constructed raw-state PPO; without this, it would remain randomly initialized.
ppo_raw_state = build_ppo(
    raw_state_adapter,
    hidden=256,
    seed=0,
)

history_raw_state = maybe_train(
    "raw-state PPO",
    ppo_raw_state,
    train_seed_offset=0,
)


# Train the newly constructed raw-state PPO directly.
# Without this step, `ppo_raw_state` would remain randomly initialized.
#history_raw_state = train_ppo_with_eval_gate(
#    system=system,
#    ppo=ppo_raw_state,
#    baseline_policy=baseline, #baseline_policy has a very specific role: it is the reference policy used only for gate evaluation, not for PPO learning itself.
#    S0=S0,
#    T=int(T),
#    total_train_episodes=int(total_train_episodes),
#    chunk_episodes=int(chunk_episodes),
#    eval_episodes=int(gate_eval_episodes),
#    eval_seed0=int(seed0) + 999,
#    train_seed0=int(seed0),
#    eval_info=eval_info,
#    verbose=True,
#)

raw_state_summary = {
    "describe": ppo_raw_state.describe_features(),
    "feature_names": ppo_raw_state.feature_names(),
    "sample_obs": ppo_raw_state._observation_vector(S0, t=0, info=None).round(3).tolist(),
}

pprint(raw_state_summary)


In [ ]:
# === Collect policies into a dictionary for easy access in evaluation/reporting code ===
policies = {
    'OrderUpTo_350': baseline,
    'ppo_raw_state': ppo_raw_state,
    'ppo_last_demand': ppo_last_demand,
    'ppo_forecast': ppo_forecast,
}

print('==== Policy Parameters ====')
policies


---

## 🪄 Experiment run and Strict-CRN policy evaluation

In this section, we evaluate all policies under identical sampled exogenous paths using strict CRN. This ensures a fair comparison and allows us to summarize and illustrate the resulting performance differences.

In [ ]:
# === Experiment run and Strict-CRN policy evaluation ===
from inventory.evaluation.notebooks.crn_runs import run_crn_mc

run = run_crn_mc(
    system=system,
    policies=policies,
    S0=S0,
    T=T,
    n_episodes=n_episodes,
    seed0=seed0,
    info=eval_info,
    print_summary=True,
)

results = run.results
rollouts = run.rollouts
totals_by_policy = run.totals_by_policy
totals_summary = run.totals_summary

In [ ]:
# === Plot total cost distribution and boxplot ===
from inventory.evaluation.plotting import plot_totals_hist_and_box
_ =plot_totals_hist_and_box(totals_by_policy)

---

## 🔬 Results, Insights & Discussion

In this section, we summarize and interpret the main findings of the experiment in a research-paper style. We first present the key results in compact form, using a result table and a main comparison plot to highlight overall performance differences.

We then discuss what these results teach us by interpreting the baseline–candidate comparison, the default–variation comparison within the candidate class, and their relevance for the underlying modeling question.

Based on this, we answer the research questions directly and conclude with implications and possible next steps.

### What are the main results?
- compact result table
- one main comparison plot
- short interpretation of who performs best
- ***Results 1***: 

### What do we learn?
- interpret baseline vs candidate
- interpret candidate default vs candidate variation
- connect result back to the modeling question
- ***Learning 1***: 

### How would we answer our reseach questions?
- ***RQ1***: 
- ***RQ2***: 

### What next steps would we plan?
- Address implications and next steps
- ***Next step1***:


---

## 🥊 Punchline

In this section, we distill the main punchline of the experiment into an exam-relevant takeaway. Think of it as the “if you remember only two things, remember these” message.


- ***Takeaway 1***: Lorem ipsum

- ***Takeaway 2***: Dolor sit 

---

# Appendix

## 🏎️ Appendix A: Dynamics of the system under policy regimes

In this section, we study the dynamics of the system under the different policy regimes. We first inspect one strict-CRN trajectory for all three policies to compare their actions and resulting system behavior. We then visualize many trajectories together with their mean paths to identify broader dynamic patterns.

In [ ]:
rollouts.keys()

In [ ]:
# === Visual dynamics for one shared *reference episode* per policy ===
from inventory.evaluation.plotting import plot_reference_episode_rollouts_grid
_ = plot_reference_episode_rollouts_grid(
    {k: rollouts[k] for k in ['OrderUpTo_350', 'ppo_raw_state']},
    #rollouts,
    figsize=(12, 14),
    show=True,
    marker="o",
 )

In [ ]:
# === Visual dynamics across many episodes (overlay: faint lines = episodes, bold line = mean) ===

n_plot_episodes = 50

rollouts_mc = system.collect_policies_crn_rollouts_mc(
    policies,
    S0,
    T=T,
    n_episodes=n_plot_episodes,
    seed0=seed0,
    info=eval_info,
 )

from inventory.evaluation.plotting import plot_rollouts_overlay_grid
_ = plot_rollouts_overlay_grid(
    rollouts_mc,
    figsize=(12, 14),
    alpha_episode=0.12,
    linewidth_episode=1.0,
    linewidth_mean=2.8,
    title_suffix=f"(CRN-MC overlay; n={n_plot_episodes})",
    show=True,
 )

---

## 📈 Appendix B: Statistical validation

In this section, we statistically compare two policies in a rigorous and structured way. We begin by defining a baseline policy (the *control*) and a candidate policy (the *treatment*).

Given the hypothesis of interest, we define the paired-delta direction, compute the paired differences, and assess their normality. We then use these paired deltas to perform both a frequentist and a Bayesian hypothesis test.

Finally, we summarize the findings and interpret their meaning in context.

**Notes**

- This appendix supports the statistical comparison of **two** policies only. Multi-policy comparisons require different assumptions and a separate method, which is not implemented here.
- The frequentist and Bayesian hypotheses may differ slightly. They are stated explicitly in their respective sections.

### ⚖️ Paired Delta analysis

Our default hypothesis of interest is that the treatment policy has a positive effect, that is, it achieves a **lower total cost** than the control policy. We therefore compute the paired deltas in the corresponding direction and examine their distribution using a normality diagnostic. This prepares the ground for the subsequent frequentist and Bayesian analyses.


In [ ]:
# === Get paired deltas summary ===
from inventory.evaluation.notebooks.reports import print_paired_delta_summary

reports = print_paired_delta_summary(
    totals_by_policy,
    baseline_name="baseline",
    higher_is_better=False,
)

In [ ]:
# === Calculate paired deltas ===
from inventory.evaluation.notebooks.reports import compute_crn_deltas

# Note: the "other_policy" argument controls which paired deltas are computed (e.g. candidate vs. candidate_var)
crn_deltas = compute_crn_deltas(
    totals_by_policy,
    #base_policy="baseline",
    #other_policy="candidate",
    base_policy="ppo_forecast",
    other_policy="OrderUpTo_350",
    plot=False,
)

In [ ]:
# === Normality diagnostics for paired deltas ===
from inventory.evaluation.deltas import normality_diagnostics
_ = normality_diagnostics(crn_deltas)

---

### ⏱️ Frequentist analysis
In this section, we analyze the policy comparison from a frequentist perspective. Based on the paired deltas, we test whether the observed performance difference is statistically significant under the hypothesis of interest.

Depending on the distribution of the paired differences, we apply an appropriate test and interpret the resulting p-value, test statistic, and confidence interval.

In [ ]:
# === Frequentist evaluation of CRN deltas ===
from inventory.evaluation.notebooks.reports import print_frequentist_report_for_crn_deltas

_ = print_frequentist_report_for_crn_deltas(crn_deltas)

### 🔬 Insights

***Frequentist Analysis***

- **Hypothesis**: Here, we formulate the frequentist null and alternative hypotheses for the paired policy comparison.

- **Results**: Here, we report the main outputs of the frequentist test, including the test statistic, p-value, and confidence interval.

- **Interpretation**: Here, we interpret the frequentist results and relate them back to the original policy question.

---

### 🎰 Bayesian analysis
In this section, we examine the policy comparison from a Bayesian perspective. Based on the paired deltas, we estimate the probability that one policy outperforms the other and quantify the uncertainty around the effect. the goal is to assess how credible the observed performance advantage is, how large the effect is likely to be, and what this implies for our decision-making problem.

This complements the frequentist view by focusing on credible effect sizes and probabilistic interpretation rather than statistical significance alone.

In [ ]:
# === Bayesian evaluation of CRN deltas ===
from inventory.evaluation.notebooks.reports import print_bayesian_report_for_crn_deltas

print_bayesian_report_for_crn_deltas(
    crn_deltas,
    better="lower",
    deltas_are="B_minus_A",
    random_state=0,
    n_draws=200_000,
    cred_level=0.95,
    delta=50.0,
    rope=0.05,
    mode="full",
)

### 🔬 Insights

***Bayesian Analysis***

- **Hypothesis**: Here, we formulate the Bayesian hypothesis of interest for the paired policy comparison.
- **Results**: Here, we summarize the posterior results, including credible intervals and the probability that one policy outperforms the other.
- **Interpretation**: Here, we interpret the Bayesian findings and discuss what they imply for the policy comparison.

---

## 🚀 Appendix C: Further variatons

In this section, you can explore further variations of the candidate policy to generate additional insights.

To proceed, copy the **Policy Definition** section, modify the policy as needed, and rerun the evaluation pipeline. If appropriate, you may also use the appendices to support a deeper analysis of the new variation.

### 🤖 🪄Additional Policy definition

In [ ]:
# === Copy and adjust 🤖 🪄Additional Policy definition section here ===

### 🪄Additional Experiment run and Strict-CRN policy evaluation

In [ ]:
# === Copy and run 🤖 🪄Additional Experiment run and Strict-CRN policy evaluationsection here ===

### 🏎️ Additional Appendix A: Dynamics of the system under policy regimes section

In [ ]:
# === Copy and run 🏎️ Appendix A: Dynamics of the system under policy regimes section here ===

### 🔍 Additional Appendix B: Statistical validation section 

In [ ]:
# === Copy and run 🔍 Appendix B: Statistical validation section here ===

### 🔬 Additional Insights

In [ ]:
# === Insert 🔬 Additional Insights here ===

---

---

## 🔮 Appendix D: Information model definition

In this section, we specify (if relevant) which information model the policy uses to anticipate future demand. We begin by formulating the forecasting task and explaining why a forecast is needed in this setting. We then describe the training data, including the features and labels used for learning.

Next, we train the selected forecaster variants and conclude with a short sanity check to ensure that the resulting predictions are reasonable before they are embedded into the policy.

In [ ]:
# === Forecasting task ===
# - what is being predicted?
# - why is forecasting needed here?

In [ ]:
# === Forecaster training data & training ===
# - where the data comes from
# - short explanation of features and labels

# - fit constant / linear / GBDT / MLP / quantile forecaster

In [ ]:
# === Forecaster sanity check ===
# - small plot or a few example predictions